# Chemprop v2 — Experiment 2: CIP R/S Featurizer (Option A — Subclass)

This notebook tests a custom atom featurizer that **replaces** the default
`ChiralTag` (`@/@@` SMILES parity) block with a **CIP R/S one-hot** in the
atom feature vector.

**Implementation strategy — Option A (subclass):**
`CIPRSAtomFeaturizer` subclasses `MultiHotAtomFeaturizer` and overrides only
`__call__`. The feature vector is **the same length as the default** (the chiral
tag block stays 5 slots wide: R | S | unspecified | CHI_OTHER-pad | unknown-pad).
The subclass computes the chiral-block start offset from the live parent data
structures, so it stays correct for any variant (v1, v2, organic).

**Key difference from Exp 1:**
- Exp 1: chiral atom feature = `@/@@` SMILES parity tag (RDKit `ChiralTag`)
- Exp 2: chiral atom feature = CIP R/S label (`_CIPCode` atom property)

**Conditions:**

| Condition | Type | Description |
|---|---|---|
| `exp_2_cip_rs_singletask` | Single-task | CIP R/S featurizer, 3 independent models |
| `exp_2_cip_rs_multitask`  | Multi-task  | CIP R/S featurizer, joint 3-target model |

**Total fits:** 1 × 3 × 5 + 5 = **20 fits**

**Hypotheses tested:**
1. R/S should now be the easiest target (inverse of Exp 1), because the atom
   features directly encode CIP R/S — the same information Morgan FP used.
2. `@/@@` should be harder than in Exp 1, because the direct ChiralTag signal
   is removed; the model must infer parity from the graph structure alone.

Part of series: `4_chemprop_exp_1.ipynb` → **`4_chemprop_exp_2.ipynb`**


## Imports

In [1]:
import gc
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch

from rdkit import Chem
from rdkit.Chem import rdchem

from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

from chemprop import data, featurizers, models, nn
from chemprop.featurizers.atom import MultiHotAtomFeaturizer

## Configuration

Identical to Exp 1 — same data, splits, batch size, and early-stopping settings.


In [2]:
INPUT_PATH  = 'data/class_all.csv'
FOLDS_PATH  = 'data/cmrt_folds.npz'
NUM_WORKERS = 0      # set >0 if multiprocessing is available
MAX_EPOCHS  = 50     # upper bound; EarlyStopping will typically stop sooner
PATIENCE    = 10     # early stopping patience
BATCH_SIZE  = 64
SEED        = 42

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

## Load data and splits

In [3]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (3858, 6)


,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R


In [4]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load pre-computed splits from a .npz file.

    Args:
        path: Path to .npz file saved by save_folds() in notebook 2.

    Returns:
        List of dicts with keys 'train', 'val', 'test' as numpy arrays
        of molecule-level integer indices into the full dataset.
    """
    archive = np.load(path)
    fold_indices = sorted(set(
        int(k.split('_')[0].replace('fold', '')) for k in archive.files))
    return [
        {
            'train': archive[f'fold{i}_train'],
            'val':   archive[f'fold{i}_val'],
            'test':  archive[f'fold{i}_test'],
        }
        for i in fold_indices
    ]

In [5]:
mol_folds = load_folds(FOLDS_PATH)
print(f'Loaded {len(mol_folds)} folds')
for i, fold in enumerate(mol_folds):
    n_tr = len(fold['train'])
    n_v  = len(fold['val'])
    n_te = len(fold['test'])
    print(f'  Fold {i}: train={n_tr:,}  val={n_v:,}  test={n_te:,}')

Loaded 5 folds
  Fold 0: train=3,478  val=190  test=190
  Fold 1: train=3,478  val=190  test=190
  Fold 2: train=3,478  val=190  test=190
  Fold 3: train=3,478  val=190  test=190
  Fold 4: train=3,478  val=190  test=190


## Target variables and label encoding

All three targets are binary and perfectly balanced (50/50). Chance baseline: 50%.


In [6]:
TARGET_LABEL_MAPS = {
    'R/S_class':  {'R': 0, 'S': 1},
    '@/@@_class': {'@': 0, '@@': 1},
    'F/L_class':  {'F': 0, 'L': 1},
}
SINGLE_TARGETS = list(TARGET_LABEL_MAPS.keys())

for col, lmap in TARGET_LABEL_MAPS.items():
    df[f'label_{col}'] = df[col].map(lmap)

print('Class distributions (all should be 50/50):')
for col in SINGLE_TARGETS:
    vc = df[f'label_{col}'].value_counts(normalize=True)
    print(f'  {col}: {vc.to_dict()}')

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


## Featurizer: CIP R/S encoding (Option A — Subclass)

### Design

`CIPRSAtomFeaturizer` subclasses `MultiHotAtomFeaturizer` and overrides only
`__call__`. Everything else (constructor, `__len__`, `v1`/`v2`/`organic`
classmethods, `num_only`) is inherited unchanged.

### Chiral tag block layout (v1 defaults)

The chiral tag block has `1 + len(chiral_tags) = 1 + 4 = 5` slots:

| Slot (relative) | Default meaning         | Exp 2 meaning        |
|---|---|---|
| 0               | CHI_UNSPECIFIED (tag=0) | CIP R                |
| 1               | CHI_TETRAHEDRAL_CW (1)  | CIP S                |
| 2               | CHI_TETRAHEDRAL_CCW (2) | unspecified / achiral|
| 3               | CHI_OTHER (tag=3)       | zeroed out           |
| 4               | unknown pad             | zeroed out           |

The total feature vector length is **identical** to the default featurizer, so
`build_mpnn()` takes `mol_graph_featurizer` and passes `d_v`/`d_e` to `BondMessagePassing`.

### CIP assignment

`Chem.AssignStereochemistry(mol, cleanIt=True, force=True)` must be called
before featurization. `_CIPCode` is only set on atoms that are genuine
stereocenters; achiral atoms have no `_CIPCode` property → slot 2 is set.

### Offset computation

`_chiral_start` is computed from the live parent dictionaries rather than
hard-coded, so the subclass is correct for v1, v2, and organic variants.


In [7]:
class CIPRSAtomFeaturizer(MultiHotAtomFeaturizer):
    """
    Subclass of MultiHotAtomFeaturizer that replaces the ChiralTag block
    with a CIP R/S one-hot encoding.

    The feature vector length is identical to the parent class.  The chiral
    tag block (5 slots for v1) is zeroed out and then rewritten as:
        slot 0 → R
        slot 1 → S
        slot 2 → unspecified / not a stereocenter
        slot 3 → (zero)
        slot 4 → (zero)

    Requires that Chem.AssignStereochemistry() has been called on the molecule
    before featurization so that the _CIPCode atom property is available.
    """

    @property
    def _chiral_start(self) -> int:
        """Index of the first slot of the chiral tag block in the feature vector."""
        # Block sizes that precede the chiral tag block:
        #   atomic_num block : 1 + len(atomic_nums)
        #   degree block     : 1 + len(degrees)
        #   formal_charge blk: 1 + len(formal_charges)
        return (
            (1 + len(self.atomic_nums))
            + (1 + len(self.degrees))
            + (1 + len(self.formal_charges))
        )

    @property
    def _chiral_block_size(self) -> int:
        """Total number of slots occupied by the chiral tag block (including pad)."""
        return 1 + len(self.chiral_tags)   # = 5 for v1

    def __call__(self, a: rdchem.Atom | None) -> np.ndarray:
        # Run the parent featurizer to populate all non-chiral features
        x = super().__call__(a)

        if a is None:
            return x

        # Zero out the entire chiral block, then write CIP R/S
        s = self._chiral_start
        x[s : s + self._chiral_block_size] = 0.0

        cip = a.GetPropsAsDict().get('_CIPCode', None)
        if cip == 'R':
            x[s] = 1.0
        elif cip == 'S':
            x[s + 1] = 1.0
        else:
            # Achiral atom, unspecified stereocenter, or CIP not assigned
            x[s + 2] = 1.0

        return x


# ── Sanity checks ─────────────────────────────────────────────────────────────
# Test molecules: (R)- and (S)-ibuprofen — CIP confirmed by RDKit.
# We deliberately avoid hardcoding "@@  means R": CIP is determined by
# atomic priority rules, not by the SMILES @ token, so we let RDKit compute
# the CIP and then assert the correct slot is set.
#
# (R)-ibuprofen: CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O  — stereocenter at atom 10
# (S)-ibuprofen: CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O   — stereocenter at atom 10

_SMI_R = 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O'   # (R)-ibuprofen
_SMI_S = 'CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O'    # (S)-ibuprofen


def _verify_featurizer():
    """
    Confirm:
    1. Feature vector length matches the default featurizer (no dim change).
    2. CIP R atom  → slot s+0 == 1, all other chiral slots == 0.
    3. CIP S atom  → slot s+1 == 1, all other chiral slots == 0.
    4. Achiral atom → slot s+2 == 1, all other chiral slots == 0.
    """
    default_feat = MultiHotAtomFeaturizer.v1()
    cip_feat     = CIPRSAtomFeaturizer.v1()

    assert len(cip_feat) == len(default_feat), (
        f"Dimension mismatch: CIPRSAtomFeaturizer has {len(cip_feat)} features, "
        f"default has {len(default_feat)}"
    )
    print(f"Feature vector length: {len(cip_feat)} (matches default ✓)")

    # Test CIP R and CIP S using ibuprofen enantiomers.
    # CIP is computed by RDKit — we do NOT assume @@ → R or @ → S.
    for smi in [_SMI_R, _SMI_S]:
        mol = Chem.MolFromSmiles(smi)
        Chem.AssignStereochemistry(mol, cleanIt=True, force=True)

        # Find the single stereocenter
        chiral_atoms = [
            a for a in mol.GetAtoms()
            if '_CIPCode' in a.GetPropsAsDict()
        ]
        assert len(chiral_atoms) == 1, f"Expected 1 stereocenter, got {len(chiral_atoms)}"
        chiral_atom = chiral_atoms[0]
        cip_actual  = chiral_atom.GetPropsAsDict()['_CIPCode']

        feat        = cip_feat(chiral_atom)
        s           = cip_feat._chiral_start
        block       = feat[s : s + cip_feat._chiral_block_size]

        print(f"  {smi}")
        print(f"    _CIPCode={cip_actual}, chiral block={block.tolist()}")

        if cip_actual == 'R':
            assert block[0] == 1.0 and sum(block) == 1.0, \
                f"R: expected slot 0 set, got {block.tolist()}"
            print(f"    ✓ R → slot 0")
        elif cip_actual == 'S':
            assert block[1] == 1.0 and sum(block) == 1.0, \
                f"S: expected slot 1 set, got {block.tolist()}"
            print(f"    ✓ S → slot 1")

    # Achiral atom — isobutane, no stereocenters
    mol_achiral = Chem.MolFromSmiles('CC(C)C')
    Chem.AssignStereochemistry(mol_achiral, cleanIt=True, force=True)
    atom_achiral = mol_achiral.GetAtomWithIdx(1)   # central carbon
    assert '_CIPCode' not in atom_achiral.GetPropsAsDict(), \
        "Expected no _CIPCode on achiral atom"
    feat_achiral = cip_feat(atom_achiral)
    s = cip_feat._chiral_start
    block_achiral = feat_achiral[s : s + cip_feat._chiral_block_size]
    print(f"  isobutane (achiral) → chiral block={block_achiral.tolist()}")
    assert block_achiral[2] == 1.0 and sum(block_achiral) == 1.0, \
        f"Achiral: expected slot 2 set, got {block_achiral.tolist()}"
    print(f"    ✓ achiral → slot 2")

    print(f"\nAll assertions passed ✓")
    print(f"  _chiral_start={cip_feat._chiral_start}, "
          f"_chiral_block_size={cip_feat._chiral_block_size}")

_verify_featurizer()

Feature vector length: 133 (matches default ✓)
  CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O
    _CIPCode=R, chiral block=[1.0, 0.0, 0.0, 0.0, 0.0]
    ✓ R → slot 0
  CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O
    _CIPCode=S, chiral block=[0.0, 1.0, 0.0, 0.0, 0.0]
    ✓ S → slot 1
  isobutane (achiral) → chiral block=[0.0, 0.0, 1.0, 0.0, 0.0]
    ✓ achiral → slot 2

All assertions passed ✓
  _chiral_start=114, _chiral_block_size=5


## Datapoints with CIP stereochemistry assignment

`_CIPCode` is only available on RDKit `Atom` objects after
`Chem.AssignStereochemistry()` has been called. We build a thin wrapper that
calls this immediately after SMILES parsing, before constructing the
`MoleculeDatapoint`.

This replaces the plain `MoleculeDatapoint.from_smi(smi, ignore_stereo=False)`
call used in Exp 1. The `ignore_stereo=False` flag (the default) is still
required so that RDKit preserves `@/@@` tokens during parsing — they are needed
for correct CIP assignment.


In [8]:
def make_datapoint_with_cip(smi: str) -> data.MoleculeDatapoint:
    """
    Parse SMILES, assign CIP stereochemistry, and return a MoleculeDatapoint.

    The CIP assignment populates the _CIPCode atom property, which
    CIPRSAtomFeaturizer reads during featurization.  ignore_stereo=False
    (the default) is required so that @ / @@ tokens are kept by RDKit.
    """
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f'RDKit could not parse SMILES: {smi}')
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    dp = data.MoleculeDatapoint(mol=mol)
    return dp


all_data_cip = [make_datapoint_with_cip(smi) for smi in df['SMILES']]
print(f'Built {len(all_data_cip):,} CIP-assigned datapoints')

# Quick spot-check: verify _CIPCode is present on stereocenters
_test_mol = all_data_cip[0].mol
_cip_atoms = [
    (a.GetIdx(), a.GetPropsAsDict().get('_CIPCode'))
    for a in _test_mol.GetAtoms()
    if '_CIPCode' in a.GetPropsAsDict()
]
print(f'Spot-check (molecule 0): stereocenter atoms with _CIPCode: {_cip_atoms}')

Built 3,858 CIP-assigned datapoints
Spot-check (molecule 0): stereocenter atoms with _CIPCode: [(8, 'S')]


In [9]:
# Instantiate the CIP R/S featurizer using v2 defaults
cip_featurizer = CIPRSAtomFeaturizer.v2()

# Inspect feature dimensions via a representative molecule
_test_smi = df['SMILES'].iloc[0]
_test_mol_raw = Chem.MolFromSmiles(_test_smi)
Chem.AssignStereochemistry(_test_mol_raw, cleanIt=True, force=True)
_mfeat = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=cip_featurizer)
_graph = _mfeat(_test_mol_raw)

print(f'Atom feature dim : {_graph.V.shape[1]}  (should match Exp 1: 72 for v1)')
print(f'Bond feature dim : {_graph.E.shape[1]}')
print(f'Nodes            : {_graph.V.shape[0]}')
print(f'CIPRSAtomFeaturizer._chiral_start      = {cip_featurizer._chiral_start}')
print(f'CIPRSAtomFeaturizer._chiral_block_size = {cip_featurizer._chiral_block_size}')

Atom feature dim : 72  (should match Exp 1: 72 for v1)
Bond feature dim : 14
Nodes            : 17
CIPRSAtomFeaturizer._chiral_start      = 51
CIPRSAtomFeaturizer._chiral_block_size = 5


In [12]:
_atom_fdim = _mfeat.atom_fdim
_bond_fdim = _mfeat.bond_fdim
_W_i_input = _atom_fdim + _bond_fdim   # BondMessagePassing: W_i = Linear(d_v + d_e, d_h)

print(f"atom_fdim  (d_v)        : {_atom_fdim}")
print(f"bond_fdim  (d_e)        : {_bond_fdim}")
print(f"W_i input  (d_v + d_e)  : {_W_i_input}  ← must match BondMessagePassing(d_v=..., d_e=...)")
print(f"V matrix shape          : {_graph.V.shape}  ← (n_atoms, atom_fdim)")
print(f"E matrix shape          : {_graph.E.shape}  ← (2*n_bonds, bond_fdim)")
assert _graph.V.shape[1] == _atom_fdim, "V col dim != atom_fdim"
assert _graph.E.shape[1] == _bond_fdim, "E col dim != bond_fdim"
print("Dimension check passed ✓")

atom_fdim  (d_v)        : 72
bond_fdim  (d_e)        : 14
W_i input  (d_v + d_e)  : 86  ← must match BondMessagePassing(d_v=..., d_e=...)
V matrix shape          : (17, 72)  ← (n_atoms, atom_fdim)
E matrix shape          : (38, 14)  ← (2*n_bonds, bond_fdim)
Dimension check passed ✓


## Model and trainer builder functions

Identical to Exp 1. Because `CIPRSAtomFeaturizer` produces vectors of the same
length as the default featurizer, `build_mpnn()` takes `mol_graph_featurizer`
as its first argument and passes `d_v` and `d_e` to `BondMessagePassing` explicitly.


In [13]:
def build_mpnn(mol_graph_featurizer, n_tasks: int = 1) -> models.MPNN:
    """
    Build a fresh Chemprop v2 MPNN for binary classification.

    d_v and d_e are read from mol_graph_featurizer so W_i is sized correctly
    for our custom atom feature dimension.

    Args:
        mol_graph_featurizer: The SimpleMoleculeMolGraphFeaturizer in use.
        n_tasks: Number of binary classification outputs.

    Returns:
        Configured chemprop.models.MPNN ready for training.
    """
    mp  = nn.BondMessagePassing(
        d_v=mol_graph_featurizer.atom_fdim,
        d_e=mol_graph_featurizer.bond_fdim,
    )
    agg = nn.MeanAggregation()
    ffn = nn.BinaryClassificationFFN(n_tasks=n_tasks)
    metric_list = [
        nn.metrics.BinaryAUROC(),
        nn.metrics.BinaryAUPRC(),
        nn.metrics.BinaryAccuracy(),
        nn.metrics.BinaryF1Score(),
    ]
    return models.MPNN(mp, agg, ffn, batch_norm=False, metrics=metric_list)


def build_trainer(fold_idx: int, condition_name: str, target_col: str) -> pl.Trainer:
    """
    Build a Lightning Trainer with EarlyStopping and ModelCheckpoint.

    Args:
        fold_idx: The current fold number (used for checkpoint naming).
        condition_name: The featurization condition (used for checkpoint naming).
        target_col: The target being predicted (used for checkpoint naming).

    Returns:
        Configured pl.Trainer.
    """
    early_stop = EarlyStopping(
        monitor='val/roc',
        patience=PATIENCE,
        mode='max',
        verbose=False,
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename=f"{condition_name}_{target_col.replace('/', '_')}_fold{fold_idx}_best",
        monitor='val/roc',
        mode='max',
        save_top_k=1,
        verbose=False,
    )
    return pl.Trainer(
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False,
        accelerator='auto',
        devices=1,
        max_epochs=MAX_EPOCHS,
        callbacks=[early_stop, checkpoint_callback],
    )

## `evaluate_chemprop` helper

Identical to Exp 1.


In [14]:
def evaluate_chemprop(
    trainer: pl.Trainer,
    mpnn: models.MPNN,
    loader,
    y_true: np.ndarray,
) -> dict:
    """
    Run prediction and compute all metrics for one fold/split.

    Args:
        trainer: Fitted pl.Trainer.
        mpnn: Fitted MPNN model.
        loader: DataLoader to run predictions on.
        y_true: 1D numpy array of true binary labels.

    Returns:
        Dict mapping metric name to scalar value.
    """
    preds = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
    probs = torch.cat(preds).squeeze().numpy()
    preds_bin = (probs > 0.5).astype(int)
    return {
        'AUROC':    roc_auc_score(y_true, probs),
        'MCC':      matthews_corrcoef(y_true, preds_bin),
        'Accuracy': accuracy_score(y_true, preds_bin),
        'F1':       f1_score(y_true, preds_bin),
        'AUPRC':    average_precision_score(y_true, probs),
    }

## Exp 2 — Single-task training loop (CIP R/S featurizer)

1 condition × 3 targets × 5 folds = **15 total fits**


In [15]:
# ── Exp 2: CIP R/S featurizer — singletask ───────────────────────────────────
SINGLE_TASK_CONDITIONS = {
    'exp_2_cip_rs_singletask': all_data_cip,
}

mol_graph_featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(
    atom_featurizer=cip_featurizer
)
all_results = []

for condition_name, all_dpoints in SINGLE_TASK_CONDITIONS.items():
    CONDITION_START = time.time()
    print(f"\n{'#'*70}")
    print(f'Condition: {condition_name}')
    print(f"{'#'*70}")

    for target_col in SINGLE_TARGETS:
        label_col = f'label_{target_col}'
        print(f"\n  Target: {target_col}\n  {'-'*60}")

        for fold_idx, fold in enumerate(mol_folds):
            FOLD_START = time.time()
            tr_pts = [all_dpoints[i] for i in fold['train']]
            va_pts = [all_dpoints[i] for i in fold['val']]
            te_pts = [all_dpoints[i] for i in fold['test']]

            y_train = df[label_col].iloc[fold['train']].values.reshape(-1, 1)
            y_val   = df[label_col].iloc[fold['val']].values.reshape(-1, 1)
            y_test  = df[label_col].iloc[fold['test']].values.reshape(-1, 1)

            for dp, yi in zip(tr_pts, y_train): dp.y = yi
            for dp, yi in zip(va_pts, y_val):   dp.y = yi
            for dp, yi in zip(te_pts, y_test):  dp.y = yi

            train_dset = data.MoleculeDataset(tr_pts, mol_graph_featurizer)
            val_dset   = data.MoleculeDataset(va_pts, mol_graph_featurizer)
            test_dset  = data.MoleculeDataset(te_pts, mol_graph_featurizer)

            train_loader = data.build_dataloader(
                train_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=True)
            val_loader = data.build_dataloader(
                val_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)
            test_loader = data.build_dataloader(
                test_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)

            mpnn    = build_mpnn(mol_graph_featurizer, n_tasks=1)
            trainer = build_trainer(fold_idx, condition_name, target_col)
            trainer.fit(mpnn, train_loader, val_loader)

            splits_to_eval = [
                ('train', train_loader, y_train.ravel()),
                ('val',   val_loader,   y_val.ravel()),
                ('test',  test_loader,  y_test.ravel()),
            ]

            for split_name, loader, y_true in splits_to_eval:
                metrics = evaluate_chemprop(trainer, mpnn, loader, y_true)
                metrics.update({
                    'fold':          fold_idx,
                    'model':         'chemprop',
                    'featurization': condition_name,
                    'target':        target_col,
                    'split':         split_name,
                    'stopped_epoch': trainer.current_epoch,
                })
                all_results.append(metrics)

                if split_name == 'test':
                    fold_elapsed = time.time() - FOLD_START
                    print(
                        f'  fold {fold_idx} (test) | '
                        f"AUROC={metrics['AUROC']:.3f}  "
                        f"MCC={metrics['MCC']:.3f}  "
                        f"Acc={metrics['Accuracy']:.3f}  "
                        f"stopped_epoch={metrics['stopped_epoch']}  "
                        f"fold_time={fold_elapsed:.1f}s"
                    )

            del mpnn, trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()

    total_elapsed = time.time() - CONDITION_START
    print(f'\nTotal time for {condition_name}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_2_cip_rs_singletask
######################################################################

  Target: R/S_class
  ------------------------------------------------------------


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best-v1.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best-v1.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best-v1.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best-v1.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt

  fold 4 (test) | AUROC=0.831  MCC=0.561  Acc=0.779  stopped_epoch=21  fold_time=93.3s

Total time for exp_2_cip_rs_singletask: 2281.8s (38.0 min)


## Exp 2 — Multi-task loop (CIP R/S featurizer)

Joint prediction of all 3 targets. **5 total fits.**


In [16]:
# ── Exp 2 multitask ───────────────────────────────────────────────────────────
print(f"\n{'#'*70}")
MULTITASK_CONDITION = 'exp_2_cip_rs_multitask'
print(f'Condition: {MULTITASK_CONDITION}')
print(f"{'#'*70}")

CONDITION_START = time.time()

for fold_idx, fold in enumerate(mol_folds):
    FOLD_START = time.time()
    print(f"\n  Fold {fold_idx}  |  "
          f"train={len(fold['train']):,}  "
          f"val={len(fold['val']):,}  "
          f"test={len(fold['test']):,}")

    tr_pts = [all_data_cip[i] for i in fold['train']]
    va_pts = [all_data_cip[i] for i in fold['val']]
    te_pts = [all_data_cip[i] for i in fold['test']]

    label_cols = [f'label_{t}' for t in SINGLE_TARGETS]
    y_train = df[label_cols].iloc[fold['train']].values
    y_val   = df[label_cols].iloc[fold['val']].values
    y_test  = df[label_cols].iloc[fold['test']].values

    for dp, yi in zip(tr_pts, y_train): dp.y = yi
    for dp, yi in zip(va_pts, y_val):   dp.y = yi
    for dp, yi in zip(te_pts, y_test):  dp.y = yi

    train_dset = data.MoleculeDataset(tr_pts, mol_graph_featurizer)
    val_dset   = data.MoleculeDataset(va_pts, mol_graph_featurizer)
    test_dset  = data.MoleculeDataset(te_pts, mol_graph_featurizer)

    train_loader = data.build_dataloader(
        train_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
    val_loader   = data.build_dataloader(
        val_dset,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
    test_loader  = data.build_dataloader(
        test_dset,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

    mpnn    = build_mpnn(mol_graph_featurizer, n_tasks=3)
    trainer = build_trainer(fold_idx, MULTITASK_CONDITION, 'all_targets')
    trainer.fit(mpnn, train_loader, val_loader)

    splits_to_eval = [
        ('train', train_loader, y_train),
        ('val',   val_loader,   y_val),
        ('test',  test_loader,  y_test),
    ]

    for split_name, loader, y_true_all in splits_to_eval:
        preds     = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
        all_probs = torch.cat(preds).numpy()

        for task_idx, target_col in enumerate(SINGLE_TARGETS):
            probs     = all_probs[:, task_idx]
            y_true    = y_true_all[:, task_idx]
            preds_bin = (probs > 0.5).astype(int)
            metrics = {
                'AUROC':    roc_auc_score(y_true, probs),
                'MCC':      matthews_corrcoef(y_true, preds_bin),
                'Accuracy': accuracy_score(y_true, preds_bin),
                'F1':       f1_score(y_true, preds_bin),
                'AUPRC':    average_precision_score(y_true, probs),
                'fold':          fold_idx,
                'model':         'chemprop',
                'featurization': MULTITASK_CONDITION,
                'target':        target_col,
                'split':         split_name,
                'stopped_epoch': trainer.current_epoch,
            }
            all_results.append(metrics)

            if split_name == 'test':
                print(
                    f"  {target_col} (test): "
                    f"AUROC={metrics['AUROC']:.3f}  "
                    f"MCC={metrics['MCC']:.3f}  "
                    f"Acc={metrics['Accuracy']:.3f}"
                )

    fold_elapsed = time.time() - FOLD_START
    print(f'  fold_time={fold_elapsed:.1f}s')

    del mpnn, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

total_elapsed = time.time() - CONDITION_START
print(f'\nTotal time for {MULTITASK_CONDITION}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_2_cip_rs_multitask
######################################################################

  Fold 0  |  train=3,478  val=190  test=190


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best-v1.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best-v1.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best-v1.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best-v1.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Use

  R/S_class (test): AUROC=1.000  MCC=0.979  Acc=0.989
  @/@@_class (test): AUROC=0.829  MCC=0.442  Acc=0.721
  F/L_class (test): AUROC=0.837  MCC=0.537  Acc=0.768
  fold_time=234.4s

Total time for exp_2_cip_rs_multitask: 973.1s (16.2 min)


## Summary across Exp 2 conditions

In [17]:
results_df = pd.DataFrame(all_results)
metric_cols  = ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC']
target_order = ['R/S_class', '@/@@_class', 'F/L_class']
condition_order = [
    'exp_2_cip_rs_singletask',
    'exp_2_cip_rs_multitask',
]

test_results = results_df[results_df['split'] == 'test']
summary = (
    test_results
    .groupby(['featurization', 'target'])[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
summary = summary.reindex([(c, t) for c in condition_order for t in target_order])
print('Test Set Performance (Mean ± std across 5 folds):')
print(summary.to_string())

Test Set Performance (Mean ± std across 5 folds):
                                     AUROC             MCC         Accuracy              F1           AUPRC        
                                      mean     std    mean     std     mean     std    mean     std    mean     std
featurization           target                                                                                     
exp_2_cip_rs_singletask R/S_class   0.9933  0.0065  0.9539  0.0375   0.9768  0.0188  0.9768  0.0187  0.9916  0.0097
                        @/@@_class  0.8161  0.0263  0.4780  0.1001   0.7389  0.0500  0.7389  0.0503  0.8182  0.0184
                        F/L_class   0.8461  0.0331  0.5610  0.0460   0.7800  0.0231  0.7821  0.0211  0.8406  0.0395
exp_2_cip_rs_multitask  R/S_class   0.9982  0.0028  0.9853  0.0094   0.9926  0.0047  0.9926  0.0047  0.9985  0.0022
                        @/@@_class  0.8096  0.0248  0.4254  0.0750   0.7126  0.0375  0.7128  0.0381  0.8140  0.0170
                      

## Hypothesis evaluation

Two comparisons:
1. **R/S should now be easiest** — CIP R/S is encoded directly in the atom features.
2. **@/@@ should be harder than in Exp 1** — the direct ChiralTag signal is removed.


In [18]:
# ── Hypothesis 1: R/S should be easier than @/@@ (inverse of Exp 1) ──────────
df_st_test = results_df[
    (results_df['featurization'] == 'exp_2_cip_rs_singletask') &
    (results_df['split'] == 'test')
]

rs_auroc = df_st_test[df_st_test['target'] == 'R/S_class']['AUROC'].mean()
at_auroc = df_st_test[df_st_test['target'] == '@/@@_class']['AUROC'].mean()

print('Hypothesis 1: R/S easier than @/@@ for Exp 2 (encodes CIP R/S directly)?')
print(f'  R/S  AUROC (singletask): {rs_auroc:.4f}')
print(f'  @/@@ AUROC (singletask): {at_auroc:.4f}')
if rs_auroc > at_auroc:
    print('  CONFIRMED: R/S > @/@@ — consistent with direct CIP R/S encoding')
else:
    print('  NOT CONFIRMED: @/@@ >= R/S')

print()

# ── Multitask vs singletask ────────────────────────────────────────────────────
df_mt_test = results_df[
    (results_df['featurization'] == 'exp_2_cip_rs_multitask') &
    (results_df['split'] == 'test')
]

print('Multitask vs singletask AUROC (test set):')
print(f"  {'Target':<15} {'Singletask':>12} {'Multitask':>12} {'Delta':>8}")
for target in target_order:
    st = df_st_test[df_st_test['target'] == target]['AUROC'].mean()
    mt = df_mt_test[df_mt_test['target'] == target]['AUROC'].mean()
    print(f"  {target:<15} {st:>12.4f} {mt:>12.4f} {mt - st:>+8.4f}")

Hypothesis 1: R/S easier than @/@@ for Exp 2 (encodes CIP R/S directly)?
  R/S  AUROC (singletask): 0.9933
  @/@@ AUROC (singletask): 0.8161
  CONFIRMED: R/S > @/@@ — consistent with direct CIP R/S encoding

Multitask vs singletask AUROC (test set):
  Target            Singletask    Multitask    Delta
  R/S_class             0.9933       0.9982  +0.0049
  @/@@_class            0.8161       0.8096  -0.0064
  F/L_class             0.8461       0.8303  -0.0158


## Per-condition performance table (test set)

In [ ]:
print(f"{'Target':<15} {'Condition':<35} {'AUROC':>7} {'MCC':>7} {'Acc':>7}")
print('-' * 74)
for target in target_order:
    for cond in condition_order:
        s = results_df[
            (results_df['target'] == target) &
            (results_df['featurization'] == cond) &
            (results_df['split'] == 'test')
        ]
        if s.empty:
            continue
        auroc = s['AUROC'].mean()
        mcc   = s['MCC'].mean()
        acc   = s['Accuracy'].mean()
        print(f'{target:<15} {cond:<35} {auroc:>7.4f} {mcc:>7.4f} {acc:>7.4f}')
    print()

## Early stopping epoch distribution

In [19]:
epoch_summary = (
    results_df[results_df['split'] == 'test']
    .groupby(['featurization', 'target'])['stopped_epoch']
    .agg(['mean', 'min', 'max'])
    .round(1)
)
print('Epochs trained before early stopping:')
print(epoch_summary.to_string())

Epochs trained before early stopping:
                                    mean  min  max
featurization           target                    
exp_2_cip_rs_multitask  @/@@_class  44.2   38   50
                        F/L_class   44.2   38   50
                        R/S_class   44.2   38   50
exp_2_cip_rs_singletask @/@@_class  42.8   39   50
                        F/L_class   32.2   21   50
                        R/S_class   16.0   15   17


## Save results for Tukey HSD

Per-fold scores saved to CSV for downstream statistical comparison alongside
Exp 0, Exp 1, and RF results.


In [20]:
output_path = '4_chemprop_exp_2_optA_results.csv'
results_df.to_csv(output_path, index=False)
print(f'\nSaved {len(results_df)} total rows to {output_path}')
print(f"  Conditions: {results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(results_df['fold'].unique().tolist())}")
print(f"  Splits:     {results_df['split'].unique().tolist()}")
print(f"  Columns:    {results_df.columns.tolist()}")


Saved 90 total rows to 4_chemprop_exp_2_optA_results.csv
  Conditions: ['exp_2_cip_rs_singletask', 'exp_2_cip_rs_multitask']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Splits:     ['train', 'val', 'test']
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target', 'split', 'stopped_epoch']
